# Efficient Similarity Detection using k-Grams, MinHash and LSH

CSL7110 — Assignment 2  
Course: Machine Learning with Big Data  
Name: Akshay Kumar  
Roll No: M25DE1028  


## Introduction

This assignment implements a scalable similarity detection pipeline using:

- k-Gram Shingling
- Jaccard Similarity
- Min-Hashing
- Locality Sensitive Hashing (LSH)

First, documents are converted into sets of k-grams. Then, Jaccard similarity is computed to measure exact similarity. To improve efficiency, MinHash is used to approximate Jaccard similarity. Finally, LSH is applied to efficiently identify highly similar document pairs.

The same methodology is later extended to the MovieLens 100K dataset for user similarity detection.

## 1. k-Gram Construction (Shingling)

To compute document similarity, each document is first converted into a set of k-grams (also called shingles).

A k-gram is a contiguous sequence of k characters (or words) extracted using a sliding window.

For a document of length n, the number of possible k-grams is:  n - k + 1

Duplicates are removed by storing k-grams in a set, since Jaccard similarity operates on sets.

In [2]:
def read_document(path):
    with open(path, 'r') as file:
        return file.read().strip()

D1 = read_document("minhash/D1.txt")
D2 = read_document("minhash/D2.txt")
D3 = read_document("minhash/D3.txt")
D4 = read_document("minhash/D4.txt")

documents = {
    "D1": D1,
    "D2": D2,
    "D3": D3,
    "D4": D4
}

print("Document Lengths:")
for name, text in documents.items():
    print(f"{name}: {len(text)}")

Document Lengths:
D1: 1749
D2: 1747
D3: 2132
D4: 1435


In [3]:
def char_kgrams(text, k):
    grams = set()
    for i in range(len(text) - k + 1):
        gram = text[i:i+k]
        grams.add(gram)
    return grams

In [4]:
char2 = {}
char3 = {}

for name, text in documents.items():
    char2[name] = char_kgrams(text, 2)
    char3[name] = char_kgrams(text, 3)

print("Unique Character k-grams:")
for name in documents:
    print(f"{name} - 2grams: {len(char2[name])}, 3grams: {len(char3[name])}")

Unique Character k-grams:
D1 - 2grams: 263, 3grams: 765
D2 - 2grams: 262, 3grams: 762
D3 - 2grams: 269, 3grams: 828
D4 - 2grams: 255, 3grams: 698


### Word 2-grams

Word-based k-grams are constructed by splitting the document into words and creating sequences of consecutive word pairs.

If a document contains w words, the total possible word 2-grams are:  w - 1

In [5]:
def word_kgrams(text, k):
    words = text.split()
    grams = set()
    
    for i in range(len(words) - k + 1):
        gram = " ".join(words[i:i+k])
        grams.add(gram)
        
    return grams

In [6]:
word2 = {}

for name, text in documents.items():
    word2[name] = word_kgrams(text, 2)

print("Unique Word 2-grams:")
for name in documents:
    print(f"{name}: {len(word2[name])}")

Unique Word 2-grams:
D1: 279
D2: 278
D3: 337
D4: 232


## 2. Exact Jaccard Similarity

Jaccard similarity between two sets A and B is defined as:

\[
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
\]

It measures the ratio of common elements to total unique elements.

In [7]:
def jaccard_similarity(setA, setB):
    intersection = len(setA.intersection(setB))
    union = len(setA.union(setB))
    return intersection / union

In [13]:
import pandas as pd
import itertools
from IPython.display import display, Markdown

pairs = list(itertools.combinations(documents.keys(), 2))

results = []

for d1, d2 in pairs:
    j2 = jaccard_similarity(char2[d1], char2[d2])
    j3 = jaccard_similarity(char3[d1], char3[d2])
    jw = jaccard_similarity(word2[d1], word2[d2])
    
    results.append({
        "Pair": f"{d1}-{d2}",
        "Char 2-gram": round(j2, 4),
        "Char 3-gram": round(j3, 4),
        "Word 2-gram": round(jw, 4)
    })

df_results = pd.DataFrame(results)
display(Markdown("### Table 1: Exact Jaccard Similarity for All Document Pairs"))
df_results

### Table 1: Exact Jaccard Similarity for All Document Pairs

,Pair,Char 2-gram,Char 3-gram,Word 2-gram
0,D1-D2,0.9811,0.9780,0.9408
1,D1-D3,0.8157,0.5804,0.1823
2,D1-D4,0.6444,0.3051,0.0302
3,D2-D3,0.8000,0.5680,0.1737
4,D2-D4,0.6413,0.3059,0.0303
5,D3-D4,0.6530,0.3121,0.0161


## 3. Min-Hashing

Min-Hashing is a technique used to efficiently approximate Jaccard similarity.

Instead of directly computing set intersection and union, multiple hash functions are applied to the elements of each set.

For each hash function, the minimum hash value of the set is recorded.

The similarity between two documents is estimated as:

\[
J_{est}(A,B) = \frac{1}{t} \sum_{i=1}^{t} I(h_i(A) = h_i(B))
\]

where t is the number of hash functions.

In [15]:
all_3grams = set()

for grams in char3.values():
    all_3grams.update(grams)

gram_to_id = {gram: idx for idx, gram in enumerate(all_3grams)}

print("Total unique 3-grams across all documents:", len(gram_to_id))

Total unique 3-grams across all documents: 1301


In [16]:
import random

def generate_hash_functions(t, m):
    hash_funcs = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_funcs.append((a, b))
    return hash_funcs

### Building MinHash Signatures

For each document and each hash function, the minimum hash value across all 3-gram IDs is computed.

This produces a signature vector of length t for each document.

The collection of signature vectors forms the signature matrix.

In [26]:
doc_ids = {}

for name in documents:
    doc_ids[name] = {gram_to_id[gram] for gram in char3[name]}

def compute_minhash_signature(doc_ids, hash_funcs, m):
    signature = []
    
    for (a, b) in hash_funcs:
        min_hash = float('inf')
        
        for x in doc_ids:
            hash_val = (a * x + b) % m
            if hash_val < min_hash:
                min_hash = hash_val
                
        signature.append(min_hash)
        
    return signature

m = 20011
t = 20

hash_funcs = generate_hash_functions(t, m)

signatures = {}

for name in documents:
    signatures[name] = compute_minhash_signature(doc_ids[name], hash_funcs, m)

print("Length of signature for D1:", len(signatures["D1"]))

def estimate_jaccard(sig1, sig2):
    matches = 0
    
    for i in range(len(sig1)):
        if sig1[i] == sig2[i]:
            matches += 1
            
    return matches / len(sig1)

approx_similarity = estimate_jaccard(signatures["D1"], signatures["D2"])

print("Estimated Jaccard Similarity (t=20):", round(approx_similarity, 4))

Length of signature for D1: 20
Estimated Jaccard Similarity (t=20): 0.95


### Experiment: Effect of Number of Hash Functions (t)

To analyze how the number of hash functions affects estimation accuracy, MinHash was computed for multiple values of t.

In [27]:
m = 20011
t_values = [20, 60, 150, 300, 600]

results_minhash = []

for t in t_values:
    hash_funcs = generate_hash_functions(t, m)
    
    sig_D1 = compute_minhash_signature(doc_ids["D1"], hash_funcs, m)
    sig_D2 = compute_minhash_signature(doc_ids["D2"], hash_funcs, m)
    
    est_sim = estimate_jaccard(sig_D1, sig_D2)
    
    results_minhash.append({
        "t": t,
        "Estimated Similarity": round(est_sim, 4)
    })

import pandas as pd
df_minhash = pd.DataFrame(results_minhash)
df_minhash

,t,Estimated Similarity
0,20,0.9500
1,60,0.9667
2,150,1.0000
3,300,0.9767
4,600,0.9700


## 4. Locality Sensitive Hashing (LSH)

Locality Sensitive Hashing (LSH) is used to efficiently identify pairs of documents whose similarity exceeds a given threshold.

Instead of comparing all document pairs, the MinHash signature matrix is divided into bands. Documents that share identical signatures within at least one band are considered candidate pairs.

The probability that two documents become a candidate pair is:

\[
f(s) = 1 - (1 - s^r)^b
\]

where:
- s = Jaccard similarity
- r = number of rows per band
- b = number of bands

In [28]:
import numpy as np

def lsh_probability(s, r, b):
    return 1 - (1 - s**r)**b

tau = 0.7

candidates = [(5,32), (8,20), (10,16), (16,10), (20,8)]

for r, b in candidates:
    prob = lsh_probability(tau, r, b)
    print(f"r={r}, b={b}, Probability at s=0.7: {round(prob,4)}")

r=5, b=32, Probability at s=0.7: 0.9972
r=8, b=20, Probability at s=0.7: 0.695
r=10, b=16, Probability at s=0.7: 0.3677
r=16, b=10, Probability at s=0.7: 0.0327
r=20, b=8, Probability at s=0.7: 0.0064


### LSH Candidate Probability for Each Document Pair

Using the selected parameters r = 8 and b = 20, the probability that each document pair will be identified as a candidate pair is computed using:

\[
f(s) = 1 - (1 - s^8)^{20}
\]

where s is the exact Jaccard similarity (character 3-grams).

In [29]:
r = 8
b = 20

lsh_results = []

for d1, d2 in pairs:
    s = jaccard_similarity(char3[d1], char3[d2])
    prob = lsh_probability(s, r, b)
    
    lsh_results.append({
        "Pair": f"{d1}-{d2}",
        "Exact Jaccard (3-gram)": round(s, 4),
        "LSH Probability": round(prob, 4)
    })

df_lsh = pd.DataFrame(lsh_results)
df_lsh

,Pair,Exact Jaccard (3-gram),LSH Probability
0,D1-D2,0.9780,1.0000
1,D1-D3,0.5804,0.2282
2,D1-D4,0.3051,0.0015
3,D2-D3,0.5680,0.1959
4,D2-D4,0.3059,0.0015
5,D3-D4,0.3121,0.0018


## 5. MinHash on MovieLens 100K Dataset

The MovieLens 100K dataset contains 943 users and 1682 movies.

For this experiment:
- Each user is represented as a set of movies they have rated.
- Ratings and timestamps are ignored.
- Exact Jaccard similarity is computed between user pairs.
- Users with similarity ≥ 0.5 are identified.

In [33]:
import pandas as pd

data_path = "ml-100k/u.data"

df = pd.read_csv(
    data_path,
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

df.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [34]:
user_movies = {}

for user_id, group in df.groupby("user_id"):
    user_movies[user_id] = set(group["movie_id"])

print("Total users:", len(user_movies))

Total users: 943


In [35]:
import itertools

def exact_user_jaccard(u1, u2):
    set1 = user_movies[u1]
    set2 = user_movies[u2]
    return len(set1 & set2) / len(set1 | set2)

similar_pairs_exact = []

users = list(user_movies.keys())

for u1, u2 in itertools.combinations(users, 2):
    sim = exact_user_jaccard(u1, u2)
    if sim >= 0.5:
        similar_pairs_exact.append((u1, u2, round(sim, 4)))

print("Number of pairs with similarity ≥ 0.5:", len(similar_pairs_exact))

Number of pairs with similarity ≥ 0.5: 10


### MinHash for User Similarity

Each user is represented as a set of movie IDs.

MinHash signatures are computed using multiple hash functions applied to movie IDs.

The estimated Jaccard similarity is obtained by comparing matching positions in the signature vectors.

In [36]:
m = 5003

def compute_user_minhash(user_set, hash_funcs, m):
    signature = []
    
    for (a, b) in hash_funcs:
        min_hash = float('inf')
        
        for movie_id in user_set:
            hash_val = (a * movie_id + b) % m
            if hash_val < min_hash:
                min_hash = hash_val
                
        signature.append(min_hash)
        
    return signature

In [37]:
import random

def generate_hash_functions(t, m):
    hash_funcs = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_funcs.append((a, b))
    return hash_funcs

t = 50
hash_funcs = generate_hash_functions(t, m)

user_signatures = {}

for user in user_movies:
    user_signatures[user] = compute_user_minhash(user_movies[user], hash_funcs, m)

print("Signature length:", len(user_signatures[1]))

Signature length: 50


In [38]:
true_pairs = set((u1, u2) for u1, u2, _ in similar_pairs_exact)

In [39]:
def estimate_user_jaccard(sig1, sig2):
    matches = 0
    for i in range(len(sig1)):
        if sig1[i] == sig2[i]:
            matches += 1
    return matches / len(sig1)

predicted_pairs = set()

users = list(user_movies.keys())

for u1, u2 in itertools.combinations(users, 2):
    est_sim = estimate_user_jaccard(user_signatures[u1], user_signatures[u2])
    if est_sim >= 0.5:
        predicted_pairs.add((u1, u2))

print("Predicted similar pairs (≥0.5):", len(predicted_pairs))

Predicted similar pairs (≥0.5): 69


In [40]:
true_positives = true_pairs.intersection(predicted_pairs)

false_positives = predicted_pairs - true_pairs

false_negatives = true_pairs - predicted_pairs

print("True Positives:", len(true_positives))
print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

True Positives: 4
False Positives: 65
False Negatives: 6


In [41]:
t = 100
hash_funcs = generate_hash_functions(t, m)

user_signatures_100 = {}

for user in user_movies:
    user_signatures_100[user] = compute_user_minhash(user_movies[user], hash_funcs, m)

print("Signature length (t=100):", len(user_signatures_100[1]))

Signature length (t=100): 100


In [42]:
predicted_pairs_100 = set()

for u1, u2 in itertools.combinations(users, 2):
    est_sim = estimate_user_jaccard(
        user_signatures_100[u1],
        user_signatures_100[u2]
    )
    
    if est_sim >= 0.5:
        predicted_pairs_100.add((u1, u2))

print("Predicted similar pairs (t=100):", len(predicted_pairs_100))

Predicted similar pairs (t=100): 22


In [43]:
true_positives_100 = true_pairs.intersection(predicted_pairs_100)

false_positives_100 = predicted_pairs_100 - true_pairs

false_negatives_100 = true_pairs - predicted_pairs_100

print("Results for t = 100")
print("True Positives:", len(true_positives_100))
print("False Positives:", len(false_positives_100))
print("False Negatives:", len(false_negatives_100))

Results for t = 100
True Positives: 5
False Positives: 17
False Negatives: 5


In [44]:
t = 200
hash_funcs = generate_hash_functions(t, m)

user_signatures_200 = {}

for user in user_movies:
    user_signatures_200[user] = compute_user_minhash(user_movies[user], hash_funcs, m)

print("Signature length (t=200):", len(user_signatures_200[1]))

Signature length (t=200): 200


In [45]:
predicted_pairs_200 = set()

for u1, u2 in itertools.combinations(users, 2):
    est_sim = estimate_user_jaccard(
        user_signatures_200[u1],
        user_signatures_200[u2]
    )
    
    if est_sim >= 0.5:
        predicted_pairs_200.add((u1, u2))

print("Predicted similar pairs (t=200):", len(predicted_pairs_200))

Predicted similar pairs (t=200): 17


In [46]:
true_positives_200 = true_pairs.intersection(predicted_pairs_200)

false_positives_200 = predicted_pairs_200 - true_pairs

false_negatives_200 = true_pairs - predicted_pairs_200

print("Results for t = 200")
print("True Positives:", len(true_positives_200))
print("False Positives:", len(false_positives_200))
print("False Negatives:", len(false_negatives_200))

Results for t = 200
True Positives: 9
False Positives: 8
False Negatives: 1


In [47]:
def evaluate_minhash(t, runs=5):
    fp_list = []
    fn_list = []
    tp_list = []
    
    for run in range(runs):
        hash_funcs = generate_hash_functions(t, m)
        
        user_signatures = {}
        for user in user_movies:
            user_signatures[user] = compute_user_minhash(
                user_movies[user], hash_funcs, m
            )
        
        predicted_pairs = set()
        
        for u1, u2 in itertools.combinations(users, 2):
            est_sim = estimate_user_jaccard(
                user_signatures[u1],
                user_signatures[u2]
            )
            
            if est_sim >= 0.5:
                predicted_pairs.add((u1, u2))
        
        tp = len(true_pairs.intersection(predicted_pairs))
        fp = len(predicted_pairs - true_pairs)
        fn = len(true_pairs - predicted_pairs)
        
        tp_list.append(tp)
        fp_list.append(fp)
        fn_list.append(fn)
    
    return (
        sum(tp_list)/runs,
        sum(fp_list)/runs,
        sum(fn_list)/runs
    )

In [48]:
tp_avg_50, fp_avg_50, fn_avg_50 = evaluate_minhash(50)

print("Average Results (t=50)")
print("Avg TP:", round(tp_avg_50,2))
print("Avg FP:", round(fp_avg_50,2))
print("Avg FN:", round(fn_avg_50,2))

Average Results (t=50)
Avg TP: 6.8
Avg FP: 162.2
Avg FN: 3.2


In [49]:
tp_avg_100, fp_avg_100, fn_avg_100 = evaluate_minhash(100)

print("Average Results (t=100)")
print("Avg TP:", round(tp_avg_100,2))
print("Avg FP:", round(fp_avg_100,2))
print("Avg FN:", round(fn_avg_100,2))

Average Results (t=100)
Avg TP: 8.0
Avg FP: 16.4
Avg FN: 2.0


In [50]:
tp_avg_200, fp_avg_200, fn_avg_200 = evaluate_minhash(200)

print("Average Results (t=200)")
print("Avg TP:", round(tp_avg_200,2))
print("Avg FP:", round(fp_avg_200,2))
print("Avg FN:", round(fn_avg_200,2))

Average Results (t=200)
Avg TP: 8.6
Avg FP: 13.2
Avg FN: 1.4


### Summary of MinHash Performance (Averaged over 5 Runs)

In [51]:
summary_results = pd.DataFrame({
    "t (Hash Functions)": [50, 100, 200],
    "Avg True Positives": [round(tp_avg_50,2), round(tp_avg_100,2), round(tp_avg_200,2)],
    "Avg False Positives": [round(fp_avg_50,2), round(fp_avg_100,2), round(fp_avg_200,2)],
    "Avg False Negatives": [round(fn_avg_50,2), round(fn_avg_100,2), round(fn_avg_200,2)]
})

summary_results

,t (Hash Functions),Avg True Positives,Avg False Positives,Avg False Negatives
0,50,6.8,162.2,3.2
1,100,8.0,16.4,2.0
2,200,8.6,13.2,1.4


In [52]:
from collections import defaultdict

def lsh_candidate_pairs(user_signatures, r, b):
    buckets = defaultdict(list)
    
    for user, signature in user_signatures.items():
        for band in range(b):
            start = band * r
            end = start + r
            
            band_signature = tuple(signature[start:end])
            bucket_key = (band, band_signature)
            
            buckets[bucket_key].append(user)
    
    candidate_pairs = set()
    
    for users_in_bucket in buckets.values():
        if len(users_in_bucket) > 1:
            for i in range(len(users_in_bucket)):
                for j in range(i+1, len(users_in_bucket)):
                    u1 = users_in_bucket[i]
                    u2 = users_in_bucket[j]
                    if u1 < u2:
                        candidate_pairs.add((u1, u2))
                    else:
                        candidate_pairs.add((u2, u1))
    
    return candidate_pairs

In [53]:
hash_funcs = generate_hash_functions(50, m)

user_signatures_50 = {}
for user in user_movies:
    user_signatures_50[user] = compute_user_minhash(
        user_movies[user], hash_funcs, m
    )

candidates_50 = lsh_candidate_pairs(user_signatures_50, r=5, b=10)

print("Number of candidate pairs (LSH, t=50):", len(candidates_50))

Number of candidate pairs (LSH, t=50): 689


In [54]:
true_pairs_06 = set()

for u1, u2 in itertools.combinations(users, 2):
    sim = exact_user_jaccard(u1, u2)
    if sim >= 0.6:
        true_pairs_06.add((u1, u2))

print("True similar pairs (≥0.6):", len(true_pairs_06))

True similar pairs (≥0.6): 3


In [55]:
tp_50_06 = true_pairs_06.intersection(candidates_50)

fp_50_06 = candidates_50 - true_pairs_06

fn_50_06 = true_pairs_06 - candidates_50

print("LSH Results (t=50, r=5, b=10, threshold=0.6)")
print("True Positives:", len(tp_50_06))
print("False Positives:", len(fp_50_06))
print("False Negatives:", len(fn_50_06))

LSH Results (t=50, r=5, b=10, threshold=0.6)
True Positives: 3
False Positives: 686
False Negatives: 0


In [56]:
hash_funcs = generate_hash_functions(100, m)

user_signatures_100 = {}

for user in user_movies:
    user_signatures_100[user] = compute_user_minhash(
        user_movies[user], hash_funcs, m
    )

print("Signature length:", len(user_signatures_100[1]))

Signature length: 100


In [57]:
candidates_100 = lsh_candidate_pairs(user_signatures_100, r=5, b=20)

print("Number of candidate pairs (LSH, t=100):", len(candidates_100))

Number of candidate pairs (LSH, t=100): 1332


In [58]:
tp_100_06 = true_pairs_06.intersection(candidates_100)

fp_100_06 = candidates_100 - true_pairs_06

fn_100_06 = true_pairs_06 - candidates_100

print("LSH Results (t=100, r=5, b=20, threshold=0.6)")
print("True Positives:", len(tp_100_06))
print("False Positives:", len(fp_100_06))
print("False Negatives:", len(fn_100_06))

LSH Results (t=100, r=5, b=20, threshold=0.6)
True Positives: 3
False Positives: 1329
False Negatives: 0


In [59]:
hash_funcs = generate_hash_functions(200, m)

user_signatures_200 = {}

for user in user_movies:
    user_signatures_200[user] = compute_user_minhash(
        user_movies[user], hash_funcs, m
    )

print("Signature length:", len(user_signatures_200[1]))

Signature length: 200


In [60]:
candidates_200_r5 = lsh_candidate_pairs(user_signatures_200, r=5, b=40)

print("Number of candidate pairs (t=200, r=5, b=40):", len(candidates_200_r5))

Number of candidate pairs (t=200, r=5, b=40): 1938


In [61]:
tp_200_r5_06 = true_pairs_06.intersection(candidates_200_r5)

fp_200_r5_06 = candidates_200_r5 - true_pairs_06

fn_200_r5_06 = true_pairs_06 - candidates_200_r5

print("LSH Results (t=200, r=5, b=40, threshold=0.6)")
print("True Positives:", len(tp_200_r5_06))
print("False Positives:", len(fp_200_r5_06))
print("False Negatives:", len(fn_200_r5_06))

LSH Results (t=200, r=5, b=40, threshold=0.6)
True Positives: 3
False Positives: 1935
False Negatives: 0


In [62]:
candidates_200_r10 = lsh_candidate_pairs(user_signatures_200, r=10, b=20)

print("Number of candidate pairs (t=200, r=10, b=20):", len(candidates_200_r10))

Number of candidate pairs (t=200, r=10, b=20): 8


In [63]:
tp_200_r10_06 = true_pairs_06.intersection(candidates_200_r10)

fp_200_r10_06 = candidates_200_r10 - true_pairs_06

fn_200_r10_06 = true_pairs_06 - candidates_200_r10

print("LSH Results (t=200, r=10, b=20, threshold=0.6)")
print("True Positives:", len(tp_200_r10_06))
print("False Positives:", len(fp_200_r10_06))
print("False Negatives:", len(fn_200_r10_06))

LSH Results (t=200, r=10, b=20, threshold=0.6)
True Positives: 2
False Positives: 6
False Negatives: 1


### Summary of LSH Performance (Threshold = 0.6)

In [64]:
lsh_summary_06 = pd.DataFrame({
    "Configuration": [
        "t=50, r=5, b=10",
        "t=100, r=5, b=20",
        "t=200, r=5, b=40",
        "t=200, r=10, b=20"
    ],
    "True Positives": [
        len(tp_50_06),
        len(tp_100_06),
        len(tp_200_r5_06),
        len(tp_200_r10_06)
    ],
    "False Positives": [
        len(fp_50_06),
        len(fp_100_06),
        len(fp_200_r5_06),
        len(fp_200_r10_06)
    ],
    "False Negatives": [
        len(fn_50_06),
        len(fn_100_06),
        len(fn_200_r5_06),
        len(fn_200_r10_06)
    ]
})

lsh_summary_06

,Configuration,True Positives,False Positives,False Negatives
0,"t=50, r=5, b=10",3,686,0
1,"t=100, r=5, b=20",3,1329,0
2,"t=200, r=5, b=40",3,1935,0
3,"t=200, r=10, b=20",2,6,1


In [65]:
true_pairs_08 = set()

for u1, u2 in itertools.combinations(users, 2):
    sim = exact_user_jaccard(u1, u2)
    if sim >= 0.8:
        true_pairs_08.add((u1, u2))

print("True similar pairs (≥0.8):", len(true_pairs_08))

True similar pairs (≥0.8): 1


In [66]:
def evaluate_lsh(candidates, true_pairs):
    tp = len(true_pairs.intersection(candidates))
    fp = len(candidates - true_pairs)
    fn = len(true_pairs - candidates)
    return tp, fp, fn

results_08 = []

configs = [
    ("t=50, r=5, b=10", candidates_50),
    ("t=100, r=5, b=20", candidates_100),
    ("t=200, r=5, b=40", candidates_200_r5),
    ("t=200, r=10, b=20", candidates_200_r10)
]

for name, cand in configs:
    tp, fp, fn = evaluate_lsh(cand, true_pairs_08)
    results_08.append((name, tp, fp, fn))

import pandas as pd

lsh_summary_08 = pd.DataFrame(
    results_08,
    columns=["Configuration", "True Positives", "False Positives", "False Negatives"]
)

lsh_summary_08

,Configuration,True Positives,False Positives,False Negatives
0,"t=50, r=5, b=10",1,688,0
1,"t=100, r=5, b=20",1,1331,0
2,"t=200, r=5, b=40",1,1937,0
3,"t=200, r=10, b=20",1,7,0
